# 第9回：前処理をPipelineにまとめる

**今日の問い：数値列とカテゴリ列を、安全に同じモデルへ入れるにはどうするか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 列型ごとの前処理をColumnTransformerで分け、Pipelineへ一体化する
- BaseEstimatorとTransformerMixinで、意味のある自作変換器を書く
- 前処理の選択肢をGridSearchCVの探索対象に含める

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- ColumnTransformer：列ごとに別の前処理を割り当てる仕組み
- 自作変換器：fit/transformを実装した独自の前処理
- get_feature_names_out：変換後の列名を取得するAPI
- パラメータ探索：前処理やモデルの設定を系統的に比較すること
- メモリキャッシュ：共通の前処理計算を使い回す仕組み

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


## なぜ「Pipeline」が必要なのか

これまで欠損を`fillna`で埋めたり、数値だけを使ったりしてきました。実データでは**数値列と
カテゴリ列（文字）が混在**し、それぞれ別の下ごしらえが要ります。

- 数値列 → 欠損を埋める＋尺度を揃える（標準化）
- カテゴリ列 → 欠損を埋める＋数値へ変換（One-Hot：各カテゴリを0/1の列にする）

これらを手作業でやると、**第6回で学んだ前処理リーク**（検証情報の漏れ）を起こしがちです。そこで
`Pipeline`と`ColumnTransformer`を使い、**前処理からモデルまでを1つの部品**にまとめます。こうすると
交差検証や予測のたびに、前処理が正しく分割の内側で学習されます。

まず数値列・カテゴリ列を決め、学習/検証に分けます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

numeric = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
categorical = ["solvent", "catalyst", "scaffold_group"]
X = df[numeric + categorical]
y = df["active"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


## TRY：列ごとの前処理を組み立てて、モデルまで繋ぐ

各部品の役割：

- `numeric_process`：数値列の下ごしらえ（欠損補完→標準化）を並べた小さなPipeline。
- `categorical_process`：カテゴリ列の下ごしらえ（欠損補完→One-Hot）。
- `ColumnTransformer`：「この列たちには数値処理、あの列たちにはカテゴリ処理」と**列ごとに担当を割り当てる**部品。
- 最後に`Pipeline([("前処理", preprocess), ("予測", ロジスティック回帰)])`で**前処理＋モデルを一体化**。

`model.fit`一発で、前処理もモデルもまとめて学習されます。


In [ ]:
numeric_process = Pipeline([
    ("欠損補完", SimpleImputer(strategy="median")),
    ("標準化", StandardScaler()),
])
categorical_process = Pipeline([
    ("欠損補完", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("数値列", numeric_process, numeric),
    ("カテゴリ列", categorical_process, categorical),
])
model = Pipeline([("前処理", preprocess), ("予測", LogisticRegression(max_iter=1000))])
model.fit(X_train, y_train)
print(classification_report(y_valid, model.predict(X_valid), target_names=["非活性", "活性"]))


### 出力の読み方

`classification_report`は、クラスごとにprecision・recall・F1と件数(support)を並べた総合成績表です。

- **活性クラスの行**を重点的に見ます（少数派で難しいため）。第8回で学んだとおり、accuracyより各クラスのrecall/precisionが実態を映します。
- 大事なのは点数そのものより、**文字列カテゴリを含む表をエラーなく1つのモデルへ通せた**こと。手作業のOne-Hotより安全で短いです。


## 未知カテゴリが来ても止まらない

本番では、学習時に無かった溶媒名が来ることがあります。`OneHotEncoder(handle_unknown="ignore")`の
おかげで、未知カテゴリでもエラーにならず予測できます。わざと存在しない溶媒名を入れて確かめます。


In [ ]:
unknown = X_valid.iloc[[0]].copy()
unknown["solvent"] = "New-Solvent"
print("未知カテゴリを含む予測:", model.predict(unknown)[0])


### 出力の読み方

エラーで止まらず予測が返れば成功です。`handle_unknown="ignore"`が無いと、未知カテゴリで例外が出て
本番が止まります。**「本番で起きうる入力」を想定して前処理を設計する**、という実務感覚が要点です。


## CORE深掘り：変換後は列が増える。その姿を見る

One-Hotはカテゴリごとに0/1の列を作るので、**列数が増えます**。`get_feature_names_out`で変換後の列名を、
`transform`で実際の数値を確認し、Pipelineの中で何が起きているかを可視化します。


In [ ]:
names = model.named_steps["前処理"].get_feature_names_out()
transformed = model.named_steps["前処理"].transform(X_train.head(3))
if hasattr(transformed, "toarray"):
    transformed = transformed.toarray()
print("元の列数:", X_train.shape[1], "→ 変換後:", transformed.shape[1])
pd.DataFrame(transformed, columns=names, index=X_train.head(3).index).iloc[:, :10].round(2)


### 出力の読み方

- **元の列数 → 変換後**で列が増えているのは、カテゴリがOne-Hotで展開されたため。列名に`カテゴリ列__solvent_EtOH`のような名前が付きます。
- 数値列は標準化され、**平均0付近・小さめの値**になっています。One-Hot列は0か1。「モデルが実際に見ている数字」はこの姿です。

## CHANGE

数値の欠損補完を`median`から`mean`へ変え、成績を比べます。変更は`SimpleImputer(strategy=...)`の**1か所だけ**。Pipelineだと変更点が1か所に集約され、実験が管理しやすくなります。


## DEEP DIVE：自作の前処理を作り、前処理も探索対象にする

sklearnに用意された変換だけでなく、**自分の化学知識を前処理として書く**ことができます。また、
「どの補完戦略が良いか」のような前処理の選択も、モデルの設定と同じく**交差検証で選べます**。


### 自作変換器：`fit`と`transform`を持つ部品を書く

`BaseEstimator, TransformerMixin`を継承し、`fit`（学習することがあれば覚える）と`transform`（変換する）を
実装すれば、**Pipelineに差し込める自分だけの前処理**になります。ここでは「分子量あたりのTPSA」と
「最適温度からの距離」を足す変換器を作ります。


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class ChemRatioFeatures(BaseEstimator, TransformerMixin):
    "分子量あたりのTPSAと、最適温度78℃からの距離を足す自作変換器。"
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X["tpsa_per_mw"] = X["tpsa"] / X["molecular_weight"].replace(0, np.nan)
        X["temp_distance"] = (X["temperature_c"] - 78).abs()
        return X

ChemRatioFeatures().fit_transform(df[["tpsa", "molecular_weight", "temperature_c"]].head()).round(3)


### 出力の読み方

元の3列に、新しい2列（`tpsa_per_mw`・`temp_distance`）が加わっています。`fit`は何も学習せず自身を返す
だけ（この変換は統計量を使わないため）。この形にしておくと、`Pipeline`へ入れて**分割の内側で**適用でき、
第11回の特徴量設計をリークなく行えます。


### 前処理の設定を`GridSearchCV`で選ぶ

`Pipeline`の各部品の設定には`前処理__数値列__欠損補完__strategy`のように**アンダースコア2つ**で
辿り着けます。この記法を使い、補完戦略（median/mean）を交差検証で比較して自動選択します。


In [ ]:
from sklearn.model_selection import GridSearchCV

grid_pipe = Pipeline([("前処理", preprocess), ("予測", LogisticRegression(max_iter=1000))])
param_grid = {"前処理__数値列__欠損補完__strategy": ["median", "mean"]}
search = GridSearchCV(grid_pipe, param_grid, cv=5, scoring="f1")
search.fit(X_train, y_train)
print("最良設定:", search.best_params_)
print("最良CV F1:", round(search.best_score_, 3))


### 出力の読み方

`best_params_`が選ばれた補完戦略、`best_score_`がそのときの交差検証F1です。ポイントは、**前処理も
モデル設定と同じ土俵で、リークなく比較・選択できる**こと。Pipelineにまとめておいたからこそ可能になります。


## APPENDIX（任意・追加演習）

Pipelineをさらに実務的に使い込みます。90分の外の自習向けです。まず`make_column_selector`で、
**列の型（数値/文字）から自動で担当を振り分ける**書き方。列名を手で並べる手間が消えます。


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_selector, make_column_transformer

auto_pre = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), make_column_selector(dtype_include="number")),
    (make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), make_column_selector(dtype_include="object")),
)
auto_model = make_pipeline(auto_pre, LogisticRegression(max_iter=1000)).fit(X_train, y_train)
print("列の型で自動振り分けした検証精度:", round(auto_model.score(X_valid, y_valid), 3))


### 出力の読み方

`make_column_selector(dtype_include="number")`が数値列を、`"object"`が文字列列を自動で拾います。列が
増減しても書き換え不要。実データで列数が多いときに効きます。`.score`は分類では既定でaccuracyを返します。


### 前処理とモデルを「まとめて」探索する

前処理の設定とモデルのハイパーパラメータを、1つの`GridSearchCV`で同時に探します。すべてPipelineの
内側なので、リークなく公平に比較できます。


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

full = Pipeline([("前処理", preprocess), ("予測", RandomForestClassifier(random_state=42))])
grid = {
    "前処理__数値列__欠損補完__strategy": ["median", "mean"],
    "予測__max_depth": [4, 6, None],
    "予測__n_estimators": [200, 300],
}
search = GridSearchCV(full, grid, cv=5, scoring="f1")
search.fit(X_train, y_train)
print("最良設定:", search.best_params_)
print("最良CV F1:", round(search.best_score_, 3))


### 出力の読み方

前処理（補完戦略）とモデル（深さ・木の本数）の**最良の組み合わせ**が一度に選ばれます。組合せは
2×3×2=12通り×5分割=60回の学習。前処理も探索対象にできるのが、Pipeline最大の利点です。


### 学習済みPipelineを保存して再利用する

選ばれた最良のPipelineを`joblib`で保存し、読み直しても同じ予測になることを確かめます。前処理ごと
保存されるので、配布先は`predict`するだけです（第15回の永続化の先取り）。


In [ ]:
import joblib
import numpy as np

path = ROOT / "workspace" / "pipeline_09.joblib"
joblib.dump(search.best_estimator_, path)
loaded = joblib.load(path)
assert np.array_equal(search.best_estimator_.predict(X_valid), loaded.predict(X_valid)), "保存前後で予測が不一致"
print("保存・読込で同じ予測:", path)


### 出力の読み方

`assert`が通れば、前処理込みのPipelineが丸ごと保存・復元できたということ。「モデルだけ保存して前処理を
忘れる」という実務で頻発する事故を、Pipeline化で防げます。


## よくある誤り

- 全データ平均で欠損補完する
- カテゴリを意味のない大小関係へ変換する
- 本番の未知カテゴリでエラーになる

## SELF-STUDY（任意・30〜60分）

- 分子量あたりのTPSAを作る自作変換器を書き、Pipelineへ組み込む
- 数値標準化の有無と補完戦略をGridSearchCVで比較する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 自作変換器に最低限必要なメソッドは何か
2. 前処理をPipelineへ入れるとリークがなぜ防げるか
3. get_feature_names_outは何に使うか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
